In [1]:
%pip install pyannote.audio
%pip install numpy==1.26

  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached googleapis_common_protos-1.73.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/893.7 kB ? eta -:--:--
   --------------------------------------- 893.7/893.7 kB 39.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/853.6 kB ? eta -:--:--
   --------------------------------------- 853.6/853.6 kB 37.0 MB/s eta 0:00:00
   -----------------------


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   ------------- -------------------------- 5.2/15.5 MB 26.5 MB/s eta 0:00:01
   ---------------------------------- ----- 13.4/15.5 MB 32.2 MB/s eta 0:00:01
   ---------------------------------------- 15.5/15.5 MB 29.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyannote-core 6.0.1 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
pyannote-metrics 4.0.0 requires numpy>=2.2.2, but you have numpy 1.26.0 which is incompatible.
scipy 1.17.1 requires numpy<2.7,>=1.26.4, but you have numpy 1.26.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")

In [3]:
# instantiate the pipeline
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  use_auth_token=HUGGING_FACE_TOKEN
)

c:\workspace\python\venv312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\didgk\.cache\torch\pyannote\models--pyannote--speaker-diarization-3.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\workspace\python\venv312\Lib\site-packages\pyannote\audio\pipelines\speaker_verification.py:43: UserWarnin

In [4]:
import torch

# cuda가 사용 가능한 경우 cuda를 사용하도록 설정
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))
    print('cuda is available')
else:
    print('cuda is not available')

cuda is not available


In [5]:
import soundfile as sf
import numpy as np

# MP3 → wav로 변환 후 넘기기
audio_path = r"C:\workspace\python\audio\싼기타_비싼기타.mp3"

# ffmpeg로 wav 변환
import subprocess
wav_path = audio_path.replace(".mp3", ".wav")
subprocess.run([
    "ffmpeg", "-i", audio_path, "-ar", "16000", "-ac", "1", wav_path, "-y"
])

# wav 파일로 실행
diarization = pipeline(wav_path)

# dump the diarization output to disk using RTTM format
with open("싼기타_비싼기타.rttm", "w", encoding='utf-8') as rttm:
    diarization.write_rttm(rttm)


c:\workspace\python\venv312\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ..\aten\src\ATen\native\ReduceOps.cpp:1760.)
  std = sequences.std(dim=-1, correction=1)


In [6]:
# RTTM을 CSV로 변환
import pandas as pd
rttm_path = "./싼기타_비싼기타.rttm"

df_rttm = pd.read_csv(
    rttm_path,      # rttm 파일 경로
    sep=' ',        # 구분자는 띄어쓰기
    header=None,    # 헤더는 없음
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4'] 
)

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN
1,SPEAKER,싼기타_비싼기타,1,7.394,3.973,NaN,NaN,SPEAKER_01,NaN,NaN
2,SPEAKER,싼기타_비싼기타,1,11.757,4.907,NaN,NaN,SPEAKER_01,NaN,NaN
3,SPEAKER,싼기타_비싼기타,1,17.224,10.645,NaN,NaN,SPEAKER_01,NaN,NaN
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
82,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN
83,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN
84,SPEAKER,싼기타_비싼기타,1,423.642,0.798,NaN,NaN,SPEAKER_00,NaN,NaN
85,SPEAKER,싼기타_비싼기타,1,424.745,3.548,NaN,NaN,SPEAKER_00,NaN,NaN


In [7]:
# start + duration을 end로 변환
df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799
1,SPEAKER,싼기타_비싼기타,1,7.394,3.973,NaN,NaN,SPEAKER_01,NaN,NaN,11.367
2,SPEAKER,싼기타_비싼기타,1,11.757,4.907,NaN,NaN,SPEAKER_01,NaN,NaN,16.664
3,SPEAKER,싼기타_비싼기타,1,17.224,10.645,NaN,NaN,SPEAKER_01,NaN,NaN,27.869
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195
...,...,...,...,...,...,...,...,...,...,...,...
82,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN,417.445
83,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN,421.230
84,SPEAKER,싼기타_비싼기타,1,423.642,0.798,NaN,NaN,SPEAKER_00,NaN,NaN,424.440
85,SPEAKER,싼기타_비싼기타,1,424.745,3.548,NaN,NaN,SPEAKER_00,NaN,NaN,428.293


In [8]:
df_rttm["number"] = None	# number 열 만들고 None으로 초기화
df_rttm.at[0, "number"] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799,0
1,SPEAKER,싼기타_비싼기타,1,7.394,3.973,NaN,NaN,SPEAKER_01,NaN,NaN,11.367,None
2,SPEAKER,싼기타_비싼기타,1,11.757,4.907,NaN,NaN,SPEAKER_01,NaN,NaN,16.664,None
3,SPEAKER,싼기타_비싼기타,1,17.224,10.645,NaN,NaN,SPEAKER_01,NaN,NaN,27.869,None
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195,None
...,...,...,...,...,...,...,...,...,...,...,...,...
82,SPEAKER,싼기타_비싼기타,1,414.474,2.971,NaN,NaN,SPEAKER_00,NaN,NaN,417.445,None
83,SPEAKER,싼기타_비싼기타,1,417.750,3.480,NaN,NaN,SPEAKER_01,NaN,NaN,421.230,None
84,SPEAKER,싼기타_비싼기타,1,423.642,0.798,NaN,NaN,SPEAKER_00,NaN,NaN,424.440,None
85,SPEAKER,싼기타_비싼기타,1,424.745,3.548,NaN,NaN,SPEAKER_00,NaN,NaN,428.293,None


In [9]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

display(df_rttm.head(10)) 

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.976,5.823,NaN,NaN,SPEAKER_01,NaN,NaN,6.799,0
1,SPEAKER,싼기타_비싼기타,1,7.394,3.973,NaN,NaN,SPEAKER_01,NaN,NaN,11.367,0
2,SPEAKER,싼기타_비싼기타,1,11.757,4.907,NaN,NaN,SPEAKER_01,NaN,NaN,16.664,0
3,SPEAKER,싼기타_비싼기타,1,17.224,10.645,NaN,NaN,SPEAKER_01,NaN,NaN,27.869,0
4,SPEAKER,싼기타_비싼기타,1,28.667,1.528,NaN,NaN,SPEAKER_01,NaN,NaN,30.195,0
5,SPEAKER,싼기타_비싼기타,1,32.419,0.747,NaN,NaN,SPEAKER_00,NaN,NaN,33.166,1
6,SPEAKER,싼기타_비싼기타,1,33.557,3.531,NaN,NaN,SPEAKER_00,NaN,NaN,37.088,1
7,SPEAKER,싼기타_비싼기타,1,37.632,3.752,NaN,NaN,SPEAKER_00,NaN,NaN,41.384,1
8,SPEAKER,싼기타_비싼기타,1,41.621,1.104,NaN,NaN,SPEAKER_00,NaN,NaN,42.725,1
9,SPEAKER,싼기타_비싼기타,1,41.655,0.764,NaN,NaN,SPEAKER_01,NaN,NaN,42.419,2


In [10]:
df_rttm_grouped = df_rttm.groupby("number").agg(
    start=pd.NamedAgg(column='start', aggfunc='min'),
    end=pd.NamedAgg(column='end', aggfunc='max'),
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,0.976,30.195,SPEAKER_01
1,32.419,42.725,SPEAKER_00
2,41.655,44.015,SPEAKER_01
3,45.798,67.088,SPEAKER_00
4,67.224,82.776,SPEAKER_01
5,84.643,102.555,SPEAKER_00
6,103.489,117.513,SPEAKER_01
7,119.737,138.667,SPEAKER_00
8,139.346,168.956,SPEAKER_01


In [11]:
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

,start,end,speaker_id,duration
0,0.976,30.195,SPEAKER_01,29.219
1,32.419,42.725,SPEAKER_00,10.306
2,41.655,44.015,SPEAKER_01,2.360
3,45.798,67.088,SPEAKER_00,21.290
4,67.224,82.776,SPEAKER_01,15.552
5,84.643,102.555,SPEAKER_00,17.912
6,103.489,117.513,SPEAKER_01,14.024
7,119.737,138.667,SPEAKER_00,18.930
8,139.346,168.956,SPEAKER_01,29.610
9,170.925,192.318,SPEAKER_00,21.393


In [12]:
df_rttm_grouped.to_csv(
    "싼기타_비싼기타_rttm.csv",
    sep=',',
    index=False
)